<div style="text-align:center; border-radius:15px; padding:15px; color:white; margin:0; font-family: 'Orbitron', sans-serif; background: #2E0249; background: #11001C; box-shadow: 0px 4px 8px rgba(0, 0, 0, 0.3); overflow:hidden; margin-bottom: 1em;">
  <div style="font-size:150%; color:#FEE100"><b>Global Ecommerce Sales Analysis and Prediction</b></div>
  <div>This notebook was created with the help of <a href="https://devra.ai/ref/kaggle" style="color:#6666FF">Devra AI</a></div>
</div>

The global ecommerce data offers an interesting snapshot of consumer behavior. It can sometimes be surprising how diverse sales metrics are across regions and categories. If you find this notebook useful, please consider upvoting it.

## Table of Contents

- [Data Loading and Preprocessing](#Data-Loading-and-Preprocessing)
- [Exploratory Data Analysis](#Exploratory-Data-Analysis)
- [Feature Engineering](#Feature-Engineering)
- [Sales Prediction Model](#Sales-Prediction-Model)
- [Conclusion](#Conclusion)

In [ ]:
# Import libraries and supress warnings
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

# For visualizations
import matplotlib
matplotlib.use('Agg')  # use agg backend for matplotlib
import matplotlib.pyplot as plt
plt.switch_backend('Agg')
%matplotlib inline

import seaborn as sns

# For prediction
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

# Set plotting style
sns.set(style='whitegrid', palette='muted', font_scale=1.2)

## Data Loading and Preprocessing

We load the global ecommerce sales dataset and inspect its structure. Note that the column `Order_Date` is provided as a string but likely represents dates, so we convert it accordingly.

In [ ]:
# Load the dataset
df = pd.read_csv('global_ecommerce_sales_dataset.csv', encoding='ascii')

# Convert Order_Date from string to datetime
df['Order_Date'] = pd.to_datetime(df['Order_Date'], errors='coerce')

# Check the first few rows
print(df.head())

# Check data types
print(df.dtypes)

### Data Cleaning and Preprocessing

We check missing values and handle them if necessary. It's important to treat date conversion errors and missing categorical values appropriately to avoid issues in analysis or modeling.

In [ ]:
# Check for missing values in the dataframe
print(df.isnull().sum())

# For the purpose of this analysis, we drop rows with missing date information
df = df.dropna(subset=['Order_Date'])

# Optionally, we could fill missing categorical values if we find any
categorical_cols = ['Country', 'Region', 'Product_Category', 'Sales_Channel', 'Payment_Method']
for col in categorical_cols:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna('Unknown')

print('Data cleaning complete. Current dataframe shape:', df.shape)

## Exploratory Data Analysis

We now explore the dataset through a series of visualizations. The aim is to uncover interesting relationships in the data.

In [ ]:
# 1. Distribution of Customer Age
plt.figure(figsize=(10,6))
sns.histplot(df['Customer_Age'], kde=True, bins=30)
plt.title('Distribution of Customer Age')
plt.xlabel('Age')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()

# 2. Sales by Product Category (Bar Plot)
plt.figure(figsize=(12,6))
sns.barplot(x='Product_Category', y='Total_Sales_USD', data=df, estimator=np.sum, ci=None)
plt.title('Total Sales USD by Product Category')
plt.xlabel('Product Category')
plt.ylabel('Total Sales (USD)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# 3. Count of Orders per Country (Pie Chart using countplot)
plt.figure(figsize=(12,6))
sns.countplot(data=df, x='Country')
plt.title('Number of Orders per Country')
plt.xlabel('Country')
plt.ylabel('Order Count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# 4. Correlation Heatmap for Numeric Columns (if four or more present)
numeric_df = df.select_dtypes(include=[np.number])
if numeric_df.shape[1] >= 4:
    plt.figure(figsize=(10,8))
    corr = numeric_df.corr()
    sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f')
    plt.title('Correlation Heatmap for Numeric Features')
    plt.tight_layout()
    plt.show()

# 5. Pair Plot for selected numeric features
sns.pairplot(numeric_df[['Quantity', 'Unit_Price_USD', 'Customer_Age', 'Total_Sales_USD']])
plt.suptitle('Pair Plot of Selected Numeric Features', y=1.02)
plt.show()

## Feature Engineering

Before building our prediction model, we perform feature engineering to convert categorical variables to numeric representations. We use one-hot encoding for simplicity. It is also important to note if you experience errors with memory usage due to many dummy variables, consider using alternative encoding or dimensionality reduction techniques.

In [ ]:
# Create a copy to avoid modifying the original dataframe
df_model = df.copy()

# One-hot encode categorical columns
df_model = pd.get_dummies(df_model, columns=categorical_cols, drop_first=True)

# Feature: Extract year, month from Order_Date (assuming seasonality may have influence)
df_model['Order_Year'] = df_model['Order_Date'].dt.year
df_model['Order_Month'] = df_model['Order_Date'].dt.month

# Drop Order_ID and Order_Date as they are less informative for prediction
df_model = df_model.drop(['Order_ID', 'Order_Date'], axis=1)

print('Feature engineering complete. Model dataframe shape:', df_model.shape)

## Sales Prediction Model

We now develop a predictor to estimate the Total Sales in USD. This might be a useful tool for forecasting or understanding the drivers of sales. We use a simple linear regression model and report the R² score as a measure of accuracy. Note that linear methods may be simplistic and further refinement (e.g., ensemble methods) could improve the performance.

In [ ]:
# Define the target and features
target = 'Total_Sales_USD'
# Exclude the target from the features
X = df_model.drop(target, axis=1)
y = df_model[target]

# Split the data into train and test datasets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Create and train the Linear Regression model
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

# Predict on the test set
y_pred = lr_model.predict(X_test)

# Evaluate the model using R^2 score
score = r2_score(y_test, y_pred)
print('Linear Regression Model R^2 Score:', score)

## Conclusion

This notebook presented a thorough analysis of the global ecommerce sales dataset. We cleaned and preprocessed the data, explored relationships through various visualizations, and developed a linear regression model to predict Total Sales. Although the model provides an initial understanding via the R² score, further analysis could include:

- Experimenting with more sophisticated models (e.g., Random Forest, Gradient Boosting).
- Performing feature selection and hyperparameter tuning to improve prediction accuracy.
- Analyzing temporal trends in more detail based on the Order_Date.

Thank you for exploring this notebook. If you found it useful, please consider upvoting.